In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------
def validate_duplicates(df):
    """
    Remove the duplicate rows from dataframe.

    Parameters
    ----------
    df: Pandas dataframe
    
    Returns
    -------
    df: Pandas dataframe without duplicate rows 
    """    
    duplicate_rows_df = df[df.duplicated()]
    if duplicate_rows_df.shape[0] > 0:
       print('Number of rows before removing:', df.count()[0])
       print('Number of duplicate rows:', duplicate_rows_df.shape[0])
       df = df.drop_duplicates()
       print('Number of rows after removing:', df.count()[0])
    else:
       print('No duplicate rows.')
    return df

def drop_na(df, threshold_NaN):
    """
    Remove the columns from dataframe containing NaN depending on threshold_NaN.

    Parameters
    ----------
    df: Pandas dataframe
    threshold_NaN: in [0, 1], from GUI
    
    Returns
    -------
    df: Pandas dataframe 
    drop_cols: list of dropped columns
    """    
    isna_stat = (df.isna().sum()/df.shape[0]).sort_values(ascending=True)
    drop_cols = []
    if isna_stat.max() > 0.0:
       drop_cols = np.array(isna_stat[isna_stat > threshold_NaN].index)
       print('Drop columns containing more than', threshold_NaN*100,'% of NaN:', drop_cols)
       df = df.drop(drop_cols, axis=1)
    else:
       print('No need to drop columns.')
    return df, drop_cols

In [ ]:
# ---------------------------------------------------------------
# Load Dataset
# ---------------------------------------------------------------
df = pd.read_csv("../datasets/waterQuality1.csv")


## Data Cleansing

In [ ]:
# ---------------------------------------------------------------
# Normalize column names
# ---------------------------------------------------------------
df.columns = df.columns.str.lower().str.strip()

# ---------------------------------------------------------------
# Replace Excel / CSV errors with NaN
# ---------------------------------------------------------------
excel_errors = ["#NUM!", "#DIV/0!", "#REF!", "#VALUE!", "#NAME?", "#NULL!", "#N/A"]
df = df.replace(excel_errors, np.nan)

# Convert all columns to numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# ---------------------------------------------------------------
# Check for duplicates
# ---------------------------------------------------------------
validate_duplicates(df)

In [ ]:
# ---------------------------------------------------------------
# Remove NaN value
# ---------------------------------------------------------------
threshold_NaN = 0.5
df, drop_cols = drop_na(df, threshold_NaN)

In [ ]:
# ---------------------------------------------------------------
# Handle missing values
# ---------------------------------------------------------------
df = df.fillna(0)

## Dataframe Display

In [ ]:
print("Dataset shape:", df.shape)
print(display(df))

## Dataframe Statistics

In [ ]:
display(df.describe().T)

## Splitting dataframe in features and targets

In [ ]:
# ---------------------------------------------------------------
# Split features and target
# ---------------------------------------------------------------
X = df.drop(columns=["is_safe"])
y = df["is_safe"]

In [ ]:
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------
# Train/test split
# ---------------------------------------------------------------
# Notes:
# - Splits dataset into training and testing sets
# - X_train, y_train ? used to train models
# - X_test, y_test ? used to evaluate model performance
# - test_size=0.25 ? 25% of data is reserved for testing
# - random_state=42 ? ensures reproducible split (same split every run)
# - Formula (conceptually):
#       Training set: 75% of samples
#       Test set: 25% of samples
#       X_train, X_test, y_train, y_test = train_test_split(X, y, ...)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [ ]:
# ---------------------------------------------------------------
# Random Forest Classifier
# ---------------------------------------------------------------
# Notes:
# - Random Forest is an ensemble method of decision trees.
# - Each tree is trained on a random bootstrap sample of the training data.
# - At each node, a random subset of features is used to find the best split.
# - Splitting criteria: Gini Impurity or Entropy
#   - Gini Impurity: G = 1 - sum(p_i^2), where p_i = proportion of class i in node
#   - Entropy: H = -sum(p_i * log2(p_i))
# - Predictions: majority vote of all trees
# - Parameters in this code:
#   - n_estimators=300 ? number of trees
#   - max_depth=None ? trees grow fully until leaves are pure
#   - random_state=42 ? ensures reproducible results
# - rf.fit(X_train, y_train) ? trains the forest
# - rf.predict(X_test) ? predicts labels for test data
# - accuracy_score & classification_report ? evaluate performance

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("===========================================")
print("Random Forest Results")
print("===========================================")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

In [ ]:
# ---------------------------------------------------------------
# 6. Logistic Regression Classifier
# ---------------------------------------------------------------
# Notes:
# - Logistic Regression is a linear model for classification.
# - Uses the sigmoid function to predict probabilities:
#       p(y=1|x) = 1 / (1 + exp(- (w^T * x + b)))
# - Predict class 1 if probability > 0.5, else class 0
# - Pipeline is used to ensure proper preprocessing:
#   1. StandardScaler ? scales features to mean=0, std=1
#   2. LogisticRegression ? fits model on scaled features
# - Parameters in this code:
#   - max_iter=20000 ? allows more iterations for solver to converge
#   - class_weight="balanced" ? penalizes majority class in imbalanced datasets
#   - solver="lbfgs" ? optimization algorithm
# - logreg_pipeline.fit(X_train, y_train) ? trains the pipeline
# - logreg_pipeline.predict(X_test) ? predicts labels for test data
# - accuracy_score & classification_report ? evaluate performance

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Pipeline: scale ? logistic regression
logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=20000,          # way more iterations
        class_weight="balanced", # penalize majority class
        solver="lbfgs"
    ))
])

logreg_pipeline.fit(X_train, y_train)

logreg_pred = logreg_pipeline.predict(X_test)

print("===========================================")
print("Logistic Regression Results")
print("===========================================")
print("Accuracy:", accuracy_score(y_test, logreg_pred))
print("\nClassification Report:\n", classification_report(y_test, logreg_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

# ---------------------------------
# CONFUSION MATRICES
# ---------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, ax=ax[0])
ax[0].set_title("Random Forest Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(y_test, logreg_pred, ax=ax[1])
ax[1].set_title("Logistic Regression Confusion Matrix")

plt.tight_layout()
plt.show()

# ---------------------------------
# ROC CURVE COMPARISON
# ---------------------------------
rf_probs = rf.predict_proba(X_test)[:, 1]
logreg_probs = logreg_pipeline.predict_proba(X_test)[:, 1]

rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
log_fpr, log_tpr, _ = roc_curve(y_test, logreg_probs)

rf_auc = auc(rf_fpr, rf_tpr)
log_auc = auc(log_fpr, log_tpr)

plt.figure(figsize=(7, 6))
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC={rf_auc:.3f})")
plt.plot(log_fpr, log_tpr, label=f"Logistic Regression (AUC={log_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

# ---------------------------------
# ACCURACY BAR CHART
# ---------------------------------
plt.figure(figsize=(7, 6))
plt.bar(["Random Forest", "Logistic Regression"], 
        [accuracy_score(y_test, rf_pred), accuracy_score(y_test, logreg_pred)])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.show()
